# spaCy: Industrial NLP & Named Entity Recognition
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/05_NLP_Embeddings/spacy_ner_text_processing.ipynb)

spaCy turns raw text into a parsed object: tokens, lemmas, parts-of-speech, dependency trees and named entities (people, orgs, money, dates...) from one fast pass.

We explore the full pipeline plus rule-based customization with EntityRuler.

In [ ]:
!pip install -q spacy

In [ ]:
!python -m spacy download en_core_web_sm -q

## 1. The pipeline in action

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")
text = ("Ajit founded Quantsmind in Pune in 2024, investing $2.5 million. "
        "The company ships quantum software and plans a Berlin office by March 2026.")
doc = nlp(text)

for token in doc[:8]:
    print(f"{token.text:<12}{token.lemma_:<12}{token.pos_:<8}{token.dep_}")

## 2. Named entities

In [ ]:
for ent in doc.ents:
    print(f"{ent.text:<18}{ent.label_:<8}{spacy.explain(ent.label_)}")

In [ ]:
from spacy import displacy
displacy.render(doc, style="ent", jupyter=True)      # colored entity spans

## 3. Dependency structure (who did what)

In [ ]:
displacy.render(nlp("Quantsmind acquired a smaller lab in Pune."),
                style="dep", jupyter=True, options={"compact": True})

## 4. Custom entities with EntityRuler

In [ ]:
ruler = nlp.add_pipe("entity_ruler", before="ner")
ruler.add_patterns([
    {"label": "PRODUCT", "pattern": [{"LOWER": "quantum"}, {"LOWER": "software"}]},
    {"label": "CODENAME", "pattern": [{"TEXT": {"REGEX": "^QX-[0-9]+"}}]},
])

doc2 = nlp("The QX-200 quantum software platform launched last week.")
[(e.text, e.label_) for e in doc2.ents]

## 5. Batch processing at scale

In [ ]:
texts = ["Report 1 about revenue in London.", "Report 2 mentions NASA and SpaceX.",
         "Report 3 covers the 2027 summit in Tokyo."] * 20
results = []
for d in nlp.pipe(texts, batch_size=32, disable=["lemmatizer"]):
    results.append([(e.text, e.label_) for e in d.ents if e.label_ in ("ORG", "GPE")])
print(results[:3])

## Cheat sheet
| Need | Tool |
|---|---|
| entities | `doc.ents`, `EntityRuler` for domain terms |
| grammar roles | `token.dep_` + displacy |
| similarity | larger models (`en_core_web_md`) have word vectors |
| transformers accuracy | `spacy-transformers` (`en_core_web_trf`) |
| speed | `nlp.pipe(..., disable=[...])` unused components |

Rule of thumb: rules catch what you can enumerate; statistical NER catches what you cannot.